# Audit Notification Dispatch System

**Purpose:** Dispatch notifications for critical operational alerts and daily operator reporting.

**Notification Types:**
1. **Critical Alerts** (Real-time):
   - Pipeline failures
   - Data quality ERROR-level threshold breaches
   - Security incidents (denied access attempts)
   - SLA violations

2. **Daily Operator Reports** (Scheduled):
   - Daily audit summary
   - SLA compliance status
   - Trends and recommendations

**Supported Channels:**
* Email (via SMTP)
* Slack/Teams webhooks
* PagerDuty/Opsgenie integration
* Custom webhooks

**Configuration:** Uses Databricks Secrets for credentials and webhook URLs

In [0]:
# Notebook parameters
dbutils.widgets.text("notification_type", "daily_summary", "Notification Type (critical_alert/daily_summary)")
dbutils.widgets.text("alert_severity", "HIGH", "Alert Severity (HIGH/MEDIUM/LOW)")
dbutils.widgets.text("alert_title", "", "Alert Title")
dbutils.widgets.text("alert_message", "", "Alert Message")
dbutils.widgets.text("alert_context", "{}", "Alert Context (JSON)")
dbutils.widgets.text("recipient_list", "data-ops-team@company.com", "Recipient List (comma-separated)")
dbutils.widgets.text("channel", "email", "Notification Channel (email/slack/webhook)")

# Retrieve parameters
notification_type = dbutils.widgets.get("notification_type")
alert_severity = dbutils.widgets.get("alert_severity")
alert_title = dbutils.widgets.get("alert_title")
alert_message = dbutils.widgets.get("alert_message")
alert_context = dbutils.widgets.get("alert_context")
recipient_list = dbutils.widgets.get("recipient_list").split(",")
channel = dbutils.widgets.get("channel")

print(f"Notification Dispatch: {notification_type} | Severity: {alert_severity} | Channel: {channel}")

In [0]:
import json
import requests
from datetime import datetime
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import smtplib

In [0]:
# Load notification configuration from Databricks Secrets
# Secrets should be configured in scope: 'lmip-notifications'

try:
    # Email configuration
    EMAIL_SMTP_SERVER = dbutils.secrets.get(scope="lmip-notifications", key="smtp_server") or "smtp.gmail.com"
    EMAIL_SMTP_PORT = int(dbutils.secrets.get(scope="lmip-notifications", key="smtp_port") or "587")
    EMAIL_FROM = dbutils.secrets.get(scope="lmip-notifications", key="email_from") or "noreply@company.com"
    EMAIL_PASSWORD = dbutils.secrets.get(scope="lmip-notifications", key="email_password") or ""
    
    # Slack webhook
    SLACK_WEBHOOK_URL = dbutils.secrets.get(scope="lmip-notifications", key="slack_webhook") or ""
    
    # Custom webhook
    CUSTOM_WEBHOOK_URL = dbutils.secrets.get(scope="lmip-notifications", key="custom_webhook") or ""
    
    print("✓ Loaded notification configuration from secrets")
except Exception as e:
    print(f"⚠️ Warning: Could not load secrets (may not be configured yet): {e}")
    print("   Using default/empty configuration")
    EMAIL_SMTP_SERVER = "smtp.gmail.com"
    EMAIL_SMTP_PORT = 587
    EMAIL_FROM = "noreply@company.com"
    EMAIL_PASSWORD = ""
    SLACK_WEBHOOK_URL = ""
    CUSTOM_WEBHOOK_URL = ""

In [0]:
def build_email_content(notification_type, alert_severity, alert_title, alert_message, context):
    """
    Build HTML email content based on notification type
    """
    
    # Severity color coding
    severity_colors = {
        "HIGH": "#dc3545",     # Red
        "MEDIUM": "#ffc107",   # Yellow
        "LOW": "#17a2b8"       # Blue
    }
    severity_color = severity_colors.get(alert_severity, "#6c757d")
    
    # Build HTML
    html = f"""
    <html>
    <head>
        <style>
            body {{ font-family: Arial, sans-serif; line-height: 1.6; color: #333; }}
            .header {{ background-color: {severity_color}; color: white; padding: 20px; }}
            .content {{ padding: 20px; }}
            .severity {{ font-weight: bold; color: {severity_color}; }}
            .timestamp {{ color: #666; font-size: 0.9em; }}
            .context {{ background-color: #f8f9fa; padding: 15px; border-left: 4px solid {severity_color}; margin-top: 20px; }}
            .footer {{ background-color: #f1f1f1; padding: 10px; text-align: center; font-size: 0.8em; color: #666; }}
        </style>
    </head>
    <body>
        <div class="header">
            <h2>{alert_title or notification_type.upper()}</h2>
        </div>
        <div class="content">
            <p class="timestamp">Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S UTC')}</p>
            <p class="severity">Severity: {alert_severity}</p>
            <p>{alert_message}</p>
            <div class="context">
                <h4>Context Details:</h4>
                <pre>{json.dumps(context, indent=2)}</pre>
            </div>
        </div>
        <div class="footer">
            <p>LMIP Audit & Monitoring System | Databricks Workspace</p>
            <p><em>This is an automated notification. Do not reply to this email.</em></p>
        </div>
    </body>
    </html>
    """
    
    return html

print("✓ Email content builder ready")

In [0]:
def send_email_notification(recipients, subject, html_content):
    """
    Send email notification via SMTP
    """
    try:
        # Create message
        msg = MIMEMultipart('alternative')
        msg['Subject'] = subject
        msg['From'] = EMAIL_FROM
        msg['To'] = ', '.join(recipients)
        
        # Attach HTML content
        html_part = MIMEText(html_content, 'html')
        msg.attach(html_part)
        
        # Send email
        if EMAIL_PASSWORD:  # Only attempt if password is configured
            with smtplib.SMTP(EMAIL_SMTP_SERVER, EMAIL_SMTP_PORT) as server:
                server.starttls()
                server.login(EMAIL_FROM, EMAIL_PASSWORD)
                server.send_message(msg)
            
            print(f"✓ Email sent successfully to {len(recipients)} recipient(s)")
            return True
        else:
            print("⚠️ Email sending skipped - SMTP credentials not configured")
            print(f"   Would have sent to: {recipients}")
            print(f"   Subject: {subject}")
            return False
            
    except Exception as e:
        print(f"✗ Failed to send email: {str(e)}")
        return False

print("✓ Email sender ready")

In [0]:
def send_slack_notification(alert_title, alert_message, alert_severity, context):
    """
    Send notification to Slack via webhook
    """
    if not SLACK_WEBHOOK_URL:
        print("⚠️ Slack webhook URL not configured")
        return False
    
    # Severity emoji
    severity_emoji = {
        "HIGH": ":rotating_light:",
        "MEDIUM": ":warning:",
        "LOW": ":information_source:"
    }
    emoji = severity_emoji.get(alert_severity, ":bell:")
    
    # Build Slack message
    slack_payload = {
        "text": f"{emoji} *{alert_title or 'LMIP Audit Alert'}*",
        "blocks": [
            {
                "type": "header",
                "text": {
                    "type": "plain_text",
                    "text": f"{emoji} {alert_title or 'LMIP Audit Alert'}"
                }
            },
            {
                "type": "section",
                "fields": [
                    {"type": "mrkdwn", "text": f"*Severity:*\n{alert_severity}"},
                    {"type": "mrkdwn", "text": f"*Time:*\n{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"}
                ]
            },
            {
                "type": "section",
                "text": {"type": "mrkdwn", "text": alert_message}
            },
            {
                "type": "section",
                "text": {"type": "mrkdwn", "text": f"```{json.dumps(context, indent=2)}```"}
            }
        ]
    }
    
    try:
        response = requests.post(SLACK_WEBHOOK_URL, json=slack_payload)
        response.raise_for_status()
        print("✓ Slack notification sent successfully")
        return True
    except Exception as e:
        print(f"✗ Failed to send Slack notification: {str(e)}")
        return False

print("✓ Slack sender ready")

In [0]:
def send_custom_webhook(alert_title, alert_message, alert_severity, context):
    """
    Send notification to custom webhook endpoint
    """
    if not CUSTOM_WEBHOOK_URL:
        print("⚠️ Custom webhook URL not configured")
        return False
    
    # Build payload
    webhook_payload = {
        "timestamp": datetime.now().isoformat(),
        "notification_type": notification_type,
        "severity": alert_severity,
        "title": alert_title,
        "message": alert_message,
        "context": context
    }
    
    try:
        response = requests.post(CUSTOM_WEBHOOK_URL, json=webhook_payload, timeout=10)
        response.raise_for_status()
        print("✓ Custom webhook notification sent successfully")
        return True
    except Exception as e:
        print(f"✗ Failed to send custom webhook: {str(e)}")
        return False

print("✓ Custom webhook sender ready")

In [0]:
# Parse alert context JSON
try:
    context = json.loads(alert_context)
except json.JSONDecodeError:
    context = {"raw_context": alert_context}

print(f"Alert context: {context}")

In [0]:
# Main notification dispatch logic
print("="*60)
print(f"DISPATCHING NOTIFICATION: {notification_type.upper()}")
print("="*60)

success_count = 0
failure_count = 0

# Build subject line
if notification_type == "critical_alert":
    subject = f"[{alert_severity}] LMIP Alert: {alert_title or 'Critical Event'}"
else:
    subject = f"LMIP Daily Audit Summary - {datetime.now().strftime('%Y-%m-%d')}"

# Build email content
html_content = build_email_content(
    notification_type, 
    alert_severity, 
    alert_title, 
    alert_message, 
    context
)

# Dispatch based on channel
if channel == "email":
    if send_email_notification(recipient_list, subject, html_content):
        success_count += 1
    else:
        failure_count += 1
        
elif channel == "slack":
    if send_slack_notification(alert_title, alert_message, alert_severity, context):
        success_count += 1
    else:
        failure_count += 1
        
elif channel == "webhook":
    if send_custom_webhook(alert_title, alert_message, alert_severity, context):
        success_count += 1
    else:
        failure_count += 1
        
else:
    print(f"✗ Unknown notification channel: {channel}")
    failure_count += 1

print("="*60)
print(f"DISPATCH COMPLETE: {success_count} succeeded, {failure_count} failed")
print("="*60)

In [0]:
# Log notification dispatch to audit table (optional)
# This creates a record of all notifications sent for compliance

from pyspark.sql import Row
import uuid

notification_log = Row(
    notification_id=str(uuid.uuid4()),
    notification_type=notification_type,
    severity=alert_severity,
    title=alert_title,
    message=alert_message[:500],  # Truncate long messages
    channel=channel,
    recipients=','.join(recipient_list),
    success=success_count > 0,
    dispatched_at=datetime.now()
)

log_df = spark.createDataFrame([notification_log])

# Display log (could be written to a notifications audit table)
log_df.display()

print("✓ Notification dispatch logged")

In [0]:
# Return status for orchestration
result = {
    "status": "success" if success_count > 0 else "failed",
    "notification_type": notification_type,
    "channel": channel,
    "succeeded": success_count,
    "failed": failure_count,
    "recipients": len(recipient_list)
}

dbutils.notebook.exit(result)